# Baga AI — поиск дублей объявлений (DINOv3)

Риелторы перезаливают одни и те же фотографии под разными объявлениями: пережатыми,
обрезанными, с водяным знаком. Парсер склеивает только **побайтово одинаковые** файлы
(sha256), поэтому такие пары он не видит.

Зачем это нужно:

- **доверие**: в карточке можно честно писать «эти фото встречаются ещё в N объявлениях»;
- **выдача**: одна квартира не должна занимать несколько мест в топе;
- **модель цены**: дубли в train и test — утечка, метрика окажется завышенной.

**Почему DINOv3, а не SigLIP.** Здесь задача не «похожий стиль», а «та же самая картинка» —
это instance retrieval. DINOv3 обучалась self-supervised на изображениях и в этой задаче
сильнее; SigLIP склеит две разные квартиры с одинаковым ремонтом. Конфигурация видов
(`f224` + `c224`, объединение `l2 → mean → l2`) взята из museum-ноутбука, где она дала 0.899.

**Настройки Kaggle:** GPU **T4**, Internet **ON**, датасет с `KrishaParser` в Input.
**Важно:** добавь вторым датасетом вывод первого ноутбука с `photo_rooms.parquet` — без него
в выборку попадут фасады и дворы, они похожи у всех соседних объявлений и будут отброшены
как типовые, а настоящих дублей может не найтись вовсе.
DINOv3 закрыта на Hugging Face — нужен секрет **HF_KEY** (Add-ons → Secrets) и принятая
лицензия модели на huggingface.co.

In [ ]:
!pip install -q -U "transformers>=4.56"

In [ ]:
import os, re, json, math, time, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
PHOTOS_PER_LISTING = 4          # больше фото на объявление -> выше шанс поймать дубль, но дольше
BATCH = 64
WORKERS = os.cpu_count()
DUP_THR = 0.88                  # порог КАНДИДАТОВ: занижен намеренно, дальше фильтрует геометрия
GEO_MIN_INLIERS = 18            # сколько точек должно уложиться в одно преобразование
MAX_MATCHES = 15                # фото, похожее на >15 объявлений, — типовое (ЖК, рендер), не улика
GROUP_MIN_INLIERS = 40          # в группы идут только надёжные рёбра: на 18 объявления сцепляются цепочкой
SHA_MAX_LISTINGS = 8            # один файл у большего числа объявлений — шаблон (планировка, логотип)
OUT = Path("/kaggle/working/artifacts"); OUT.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}

def find_photos_root(root="/kaggle/input"):
    for dp, dn, _ in os.walk(root):
        if Path(dp).name == "photos" and sum(d.isdigit() for d in dn) > 0:
            return Path(dp)
    raise FileNotFoundError("папка photos/<listing_id>/ не найдена — добавь датасет в Input")

def find_file(name, root="/kaggle/input"):
    hits = [Path(dp) / name for dp, _, fn in os.walk(root) if name in fn]
    return hits[0] if hits else None

PHOTOS_ROOT = find_photos_root()
DB = find_file("krisha.db")
ROOMS = find_file("photo_rooms.parquet")      # из первого ноутбука, необязательно

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
print("photos:", PHOTOS_ROOT, "\nDB:", DB, "\nтипы комнат:", ROOMS or "нет (возьму первые фото)")
print("device:", device)
if device == "cpu":
    print("\n!!! GPU НЕ ВКЛЮЧЁН — Session options -> Accelerator -> GPU T4")

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_KEY"))
    print("вход в Hugging Face выполнен")
except Exception as e:
    print("нет секрета HF_KEY:", e, "\nDINOv3 закрыта — без токена модель не скачается")

## 1. Какие фото берём

Дубли ищем по интерьерам: фасады и подъезды одинаковы у всех соседних объявлений и дали бы
ложные склейки. Если рядом лежит `photo_rooms.parquet` из первого ноутбука, берём интерьерные
и не-рендеры; иначе просто первые фото каждого объявления.

Пары, где файл физически один и тот же (sha256-дедуп парсера), известны заранее — добавим их
в результат как достоверные.

In [ ]:
REL = re.compile(r"(\d+)[\\/]+(\d+\.jpg)$")
INTERIOR = {"kitchen", "living_room", "bedroom", "bathroom", "hallway", "balcony"}

con = sqlite3.connect(f"file:{DB}?mode=ro&immutable=1", uri=True)
photos = pd.read_sql("SELECT listing_id, idx AS photo_n, local_path, sha256 FROM photos "
                     "WHERE local_path IS NOT NULL", con)
photos["path"] = [f"{m[1]}/{m[2]}" if (m := REL.search(p or "")) else None for p in photos.local_path]
photos = photos.dropna(subset=["path"])
photos["listing_id"] = photos.listing_id.astype(str)
photos = photos[[(PHOTOS_ROOT / p).exists() for p in photos.path]]
print("строк фото:", len(photos), "| объявлений:", photos.listing_id.nunique())

# пары, где файл буквально один и тот же
shared = photos.dropna(subset=["sha256"]).groupby("sha256").listing_id.nunique()
usable = shared[(shared > 1) & (shared <= SHA_MAX_LISTINGS)].index
sha_pairs = set()
for _, grp in photos[photos.sha256.isin(usable)].groupby("sha256"):
    ids = sorted(grp.listing_id.unique())
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            sha_pairs.add((ids[i], ids[j]))
print("пар с общим файлом (sha256):", len(sha_pairs))
mass = shared[shared > SHA_MAX_LISTINGS]
if len(mass):
    print(f"файлов-шаблонов пропущено: {len(mass)}, самый частый — у {int(mass.max())} объявлений")

if ROOMS is not None:
    rooms = pd.read_parquet(ROOMS)
    rooms["listing_id"] = rooms.listing_id.astype(str)
    good = rooms[rooms.room_type.isin(INTERIOR) & (rooms.render_prob < 0.7)][["listing_id", "photo_n"]]
    photos = photos.merge(good, on=["listing_id", "photo_n"], how="inner")
    print("после фильтра интерьеров:", len(photos))

sub = (photos.sort_values(["listing_id", "photo_n"])
       .groupby("listing_id", sort=False).head(PHOTOS_PER_LISTING).reset_index(drop=True))
files = [str(PHOTOS_ROOT / p) for p in sub.path]
print(f"к прогону: {len(sub)} фото из {sub.listing_id.nunique()} объявлений")

## 2. DINOv3: препроцессинг и модель

Виды `f224` (растяжение в квадрат) и `c224` (центральный кроп), объединение `l2 → mean → l2` —
конфигурация из museum.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

proc = AutoImageProcessor.from_pretrained(MODEL_ID)
# ВАЖНО: модель держим в fp32 и считаем через autocast. Если перевести ViT-L целиком
# в fp16 (.to(dtype=torch.float16)), активации переполняются, получается inf, а после
# нормализации — NaN, и все косинусы молча становятся ложными. В museum это было сделано
# правильно: dorewka считал в fp32, multires — через autocast.
model = AutoModel.from_pretrained(MODEL_ID).eval().to(device)
norm = T.Normalize(proc.image_mean, proc.image_std)
VIEWS = {
    "f224": T.Compose([T.Resize((224, 224)), T.ToTensor(), norm]),
    "c224": T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm]),
}

class PhotoDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as raw:
                im = ImageOps.exif_transpose(raw).convert("RGB")
            return {k: tf(im) for k, tf in VIEWS.items()}, i, True
        except Exception:
            return {k: torch.zeros(3, 224, 224) for k in VIEWS}, i, False

@torch.inference_mode()
def encode(xs):
    acc = None
    for x in xs.values():
        with torch.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
            out = model(pixel_values=x.to(device)).pooler_output
        v = F.normalize(out.float(), dim=-1)
        acc = v if acc is None else acc + v
    return F.normalize(acc, dim=-1).cpu().numpy()

dl = DataLoader(PhotoDS(files[:BATCH * 3]), batch_size=BATCH, num_workers=WORKERS, pin_memory=True)
t0 = time.time()
for xs, _, _ in dl:
    v = encode(xs)
dt = (time.time() - t0) / 3
D = v.shape[1]
print(f"dim={D} | {dt:.1f} с/батч -> прогноз {dt * len(files) / BATCH / 60:.0f} мин на {len(files)} фото")

# Проверка, которой не хватало в прошлый раз: NaN в эмбеддингах ломает ВСЕ косинусы молча.
assert np.isfinite(v).all(), "в эмбеддингах NaN/inf — модель считает в fp16 без autocast?"
print(f"проверка: все значения конечны, норма вектора {np.linalg.norm(v[0]):.3f} (должна быть 1.0)")

## 3. Эмбеддинги (шардами, можно прерывать)

In [ ]:
SHARD = 4096
CKPT = Path("/kaggle/working/dino_shards"); CKPT.mkdir(exist_ok=True)

n_sh = math.ceil(len(files) / SHARD)
for s in tqdm(range(n_sh), desc="шарды"):
    f = CKPT / f"shard_{s:05d}.npz"
    if f.exists():
        continue
    chunk = files[s * SHARD:(s + 1) * SHARD]
    dl = DataLoader(PhotoDS(chunk), batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=True)
    E = np.zeros((len(chunk), D), np.float32); ok = np.zeros(len(chunk), bool)
    for xs, ii, good in tqdm(dl, leave=False):
        E[ii.numpy()] = encode(xs); ok[ii.numpy()] = good.numpy()
    tmp = CKPT / f"shard_{s:05d}.tmp.npz"
    np.savez(tmp, emb=E.astype(np.float16), valid=ok); os.replace(tmp, f)

parts = [np.load(CKPT / f"shard_{s:05d}.npz") for s in range(n_sh)]
E = np.concatenate([p["emb"] for p in parts]).astype(np.float32)
valid = np.concatenate([p["valid"] for p in parts])
bad = int((~np.isfinite(E)).any(1).sum())
print("эмбеддинги:", E.shape, "| битых фото:", int((~valid).sum()), "| строк с NaN/inf:", bad)
if bad:
    raise RuntimeError(f"{bad} строк с NaN — удали /kaggle/working/dino_shards и пересчитай: "
                       "старые шарды посчитаны сломанной версией")
print("нормы векторов: min", float(np.linalg.norm(E, axis=1).min()).__round__(3),
      "max", float(np.linalg.norm(E, axis=1).max()).__round__(3))

## 4. Поиск пар

Считаем косинусы чанками. Три правила, чтобы не наловить мусора:

- фото одного и того же объявления между собой не сравниваем;
- фото, похожее больше чем на `MAX_MATCHES` объявлений, выбрасываем: это типовой снимок
  (двор, ЖК, шаблонный рендер), а не улика;
- для пары объявлений храним максимальный косинус и число совпавших фото — по двум
  совпавшим фото уверенность выше, чем по одному.

In [ ]:
lid = sub.listing_id.to_numpy()
pair_score, pair_count = {}, {}
best_cross = np.zeros(len(E), dtype=np.float32)   # для диагностики: насколько похоже ближайшее ЧУЖОЕ фото
n_generic = 0

for a in tqdm(range(0, len(E), 1024), desc="сравнение"):
    S = E[a:a + 1024] @ E.T
    S[:, ~valid] = -1
    S[~valid[a:a + 1024]] = -1
    S[lid[a:a + 1024, None] == lid[None, :]] = -1      # своё же объявление
    best_cross[a:a + 1024] = S.max(1)

    hits = S >= DUP_THR
    generic = np.flatnonzero(hits.sum(1) > MAX_MATCHES)  # типовое фото: двор, фасад, шаблонный рендер
    n_generic += len(generic)
    hits[generic] = False
    for r, c in zip(*np.nonzero(hits)):
        key = tuple(sorted((lid[a + r], lid[c])))
        pair_score[key] = max(pair_score.get(key, 0.0), float(S[r, c]))
        pair_count[key] = pair_count.get(key, 0) + 1

pairs = pd.DataFrame(list(pair_score.items()), columns=["pair", "cos"])
if len(pairs):
    pairs[["a", "b"]] = pd.DataFrame(pairs.pair.tolist(), index=pairs.index)
    pairs["n_photos"] = [pair_count[k] // 2 or 1 for k in pairs.pair]
    pairs = pairs.drop(columns="pair").sort_values("cos", ascending=False)
else:
    pairs = pd.DataFrame(columns=["a", "b", "cos", "n_photos"])      # пустой, но с колонками

print(f"пар-кандидатов при пороге {DUP_THR}: {len(pairs)}")
print(f"фото отброшено как типовые (похожи более чем на {MAX_MATCHES} объявлений): {n_generic}")
print("из них уже известны по sha256:", sum((a, b) in sha_pairs for a, b in zip(pairs.a, pairs.b)))

# Главная диагностика: распределение похожести на ближайшее ЧУЖОЕ фото.
# По ней виден правильный порог: дубли лежат в правом хвосте.
print("\nближайшее фото из другого объявления, перцентили косинуса:")
for q in (50, 90, 99, 99.9, 99.99):
    print(f"  {q:6.2f}%: {np.percentile(best_cross, q):.3f}")
print(f"  максимум: {best_cross.max():.3f}")
print(f"\nфото с косинусом выше порога {DUP_THR}: {int((best_cross >= DUP_THR).sum())}")
if len(pairs) == 0:
    print("\n!!! ПАР НЕТ. Что делать:")
    print("  - подключи photo_rooms.parquet из первого ноутбука (тогда уйдут фасады и дворы),")
    print("  - или опусти DUP_THR к значению 99.9-го перцентиля из таблицы выше,")
    print("  - или подними MAX_MATCHES: типовых фото могло оказаться слишком много.")
pairs.head(10)

## 5. Геометрическая проверка: та же сцена или просто похожая

Косинус говорит «похоже», но не отличает одну и ту же комнату от двух разных квартир
с одинаковым ремонтом. Геометрия отличает.

На обоих фото находим ключевые точки (SIFT), ищем соответствия, прогоняем RANSAC и считаем,
сколько точек уложилось в одно преобразование (гомографию). У одной и той же сцены таких
точек десятки, у просто похожих интерьеров — единицы, потому что случайные совпадения
не складываются в согласованную геометрию.

Это позволяет держать порог косинуса низким (0.88) и ловить больше пережатых копий:
ложные склейки отсекутся здесь, а не порогом.

In [ ]:
import cv2

sift = cv2.SIFT_create(nfeatures=1500)
matcher = cv2.BFMatcher()
_kp_cache = {}


def keypoints(path, max_side=640):
    """Дескрипторы одного фото. Кэшируем: одно фото участвует в нескольких парах."""
    if path in _kp_cache:
        return _kp_cache[path]
    img = cv2.imread(str(PHOTOS_ROOT / path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        _kp_cache[path] = (None, None)
        return None, None
    k = max_side / max(img.shape)
    if k < 1:
        img = cv2.resize(img, (int(img.shape[1] * k), int(img.shape[0] * k)), interpolation=cv2.INTER_AREA)
    kp, des = sift.detectAndCompute(img, None)
    _kp_cache[path] = (kp, des)
    return kp, des


def geometric_inliers(path_a, path_b) -> int:
    """Сколько точек уложилось в одну гомографию. 0 — сцены разные."""
    kp1, des1 = keypoints(path_a)
    kp2, des2 = keypoints(path_b)
    if des1 is None or des2 is None or len(kp1) < 8 or len(kp2) < 8:
        return 0
    good = [m for m, n in matcher.knnMatch(des1, des2, k=2) if m.distance < 0.75 * n.distance]  # тест Лоу
    if len(good) < 8:
        return 0
    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    _, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    return int(mask.sum()) if mask is not None else 0


# для каждой пары объявлений проверяем геометрией самую похожую пару фото
best_photo_pair = {}
for a in tqdm(range(0, len(E), 1024), desc="лучшие пары фото"):
    S = E[a:a + 1024] @ E.T
    S[:, ~valid] = -1
    S[~valid[a:a + 1024]] = -1
    S[lid[a:a + 1024, None] == lid[None, :]] = -1
    for r, c in zip(*np.nonzero(S >= DUP_THR)):
        key = tuple(sorted((lid[a + r], lid[c])))
        if S[r, c] > best_photo_pair.get(key, (0, None, None))[0]:
            best_photo_pair[key] = (float(S[r, c]), sub.path.iat[a + r], sub.path.iat[c])

print(f"пар-кандидатов при косинусе >= {DUP_THR}: {len(best_photo_pair)}")

rows = []
for (a, b), (cos, pa, pb) in tqdm(best_photo_pair.items(), desc="геометрия"):
    inl = geometric_inliers(pa, pb)
    rows.append({"a": a, "b": b, "cos": cos, "inliers": inl,
                 "n_photos": max(1, pair_count.get((a, b), 2) // 2), "photo_a": pa, "photo_b": pb})

cand = pd.DataFrame(rows, columns=["a", "b", "cos", "inliers", "n_photos", "photo_a", "photo_b"])
cand = cand.astype({"cos": float, "inliers": int, "n_photos": int})   # иначе пустой кадр даёт object
cand = cand.sort_values(["inliers", "cos"], ascending=False)
pairs = cand[cand.inliers >= GEO_MIN_INLIERS].copy()
print(f"прошло геометрию (>= {GEO_MIN_INLIERS} точек): {len(pairs)} из {len(cand)}")
if len(cand) == 0:
    print("!!! кандидатов нет — смотри диагностику порога в предыдущей ячейке")
print("\nсколько пар остаётся при разных требованиях к геометрии:")
for thr in (8, 12, 18, 25, 40):
    print(f"  >= {thr:2d} точек: {int((cand.inliers >= thr).sum()):4d} пар")
cand.head(8)[["a", "b", "cos", "inliers", "n_photos"]]

## 6. Порог — подбираем глазами

Теперь решают два числа: косинус (кандидаты) и число точек геометрии (подтверждение).
Смотрим глазами на пары у границы — и на те, что косинус посчитал похожими, а геометрия
отвергла: это как раз разные квартиры с одинаковым ремонтом.

In [ ]:
for thr in (0.88, 0.90, 0.92, 0.95, 0.97):
    n = int((pairs.cos >= thr).sum())
    print(f"косинус >= {thr}: пар {n:5d} (все они уже прошли геометрию)")

def show_pair(row, n=3):
    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 5))
    for ax in axes.ravel():
        ax.axis("off")
    for row_i, listing in enumerate((row.a, row.b)):
        paths = sub[sub.listing_id == listing].path.tolist()[:n]
        for ax, p in zip(axes[row_i], paths):
            with Image.open(PHOTOS_ROOT / p) as im:
                im = ImageOps.exif_transpose(im).convert("RGB"); im.thumbnail((400, 400))
                ax.imshow(im)
        axes[row_i][0].set_title(f"{listing}", fontsize=9, loc="left")
    fig.suptitle(f"cos={row.cos:.3f}, точек геометрии: {getattr(row, 'inliers', '—')}, совпало фото: {row.n_photos}")
    plt.tight_layout(); plt.show()

if len(pairs) == 0 and len(cand) == 0:
    print("пар нет — показывать нечего, смотри диагностику выше")
else:
    if len(pairs):
        print("\nсамые уверенные пары:")
        for r in pairs.head(3).itertuples():
            show_pair(r)
        print("\nпары у границы — по ним решаем, где ставить порог:")
        for r in pairs.sort_values("inliers").head(3).itertuples():
            show_pair(r)
    rejected = cand[cand.inliers < GEO_MIN_INLIERS] if len(cand) else cand
    if len(rejected):
        print("\nОТВЕРГНУТЫЕ геометрией (косинус высокий, сцена другая) — проверь, что это разные квартиры:")
        for r in rejected.sort_values("cos", ascending=False).head(3).itertuples():
            show_pair(r)

## 6. Группы дублей и сохранение

In [ ]:
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x

# В pairs намеренно лежат и слабые совпадения — порог занижен, чтобы его можно было
# подобрать потом. В граф групп идут только надёжные: по слабым рёбрам объявления с
# типовым ремонтом от застройщика сцепляются в одну длинную цепочку.
strong = pairs[pairs.inliers >= GROUP_MIN_INLIERS]
print(f"рёбер в граф групп: {len(strong)} из {len(pairs)} (inliers >= {GROUP_MIN_INLIERS})")
for a, b in list(sha_pairs) + list(zip(strong.a, strong.b)):
    parent[find(a)] = find(b)

groups = pd.DataFrame({"listing_id": list(parent)})
groups["dup_group"] = [find(x) for x in groups.listing_id]
sizes = groups.groupby("dup_group").size()
groups = groups[groups.dup_group.isin(sizes[sizes > 1].index)]

# сколько чужих объявлений делят фото с этим — сигнал доверия для карточки
if len(strong):
    shared_with = pd.concat([strong[["a", "b"]], strong[["b", "a"]].rename(columns={"b": "a", "a": "b"})])
    trust = shared_with.groupby("a").b.nunique().rename("shared_photo_listings").reset_index()
    trust.columns = ["listing_id", "shared_photo_listings"]
else:
    trust = pd.DataFrame(columns=["listing_id", "shared_photo_listings"])

np.save(OUT / "dino_emb.npy", E.astype(np.float16))          # пригодятся для «качества ремонта»
sub.to_parquet(OUT / "dino_index.parquet", index=False)
groups.to_parquet(OUT / "listing_dupes.parquet", index=False)
pairs.to_csv(OUT / "dupe_pairs.csv", index=False)
trust.to_parquet(OUT / "listing_trust.parquet", index=False)
big = groups.groupby("dup_group").size().sort_values(ascending=False)
print(f"групп дублей: {len(big)}, объявлений в них: {len(groups)}, размеры топ-5: {list(big.head(5))}")
if len(big) and big.max() > 20:
    print(f"!!! группа из {big.max()} объявлений — столько одинаковых квартир не бывает."
          " Подними GROUP_MIN_INLIERS или SHA_MAX_LISTINGS.")
print(f"объявлений с чужими фото: {len(trust)}")
print("\nсохранено:", *[p.name for p in OUT.iterdir()])

## Дальше — локально

Скачай из Output папку `artifacts/` и положи в `baga-ai/artifacts/`. Дальше эти файлы
подключаются к трём местам:

- **карточка на сайте** — значок «эти фото встречаются ещё в N объявлениях» (`listing_trust.parquet`);
- **выдача поиска** — схлопывание объявлений одной группы в одно (`listing_dupes.parquet`);
- **модель цены** — разбиение train/test по `dup_group`, чтобы одна квартира не попала
  в обе части.